# 01 — Data loading and QC

Read config/paths.yaml and config/analysis_params.yaml. Download and arrange GSE304669 with `python -m src.prepare_data --archive data/GSE304669_RAW.tar`. Set SKR_RUN_DIR to a new directory for each complete run. Counts and log-normalized expression are kept separately.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from src import preprocessing, utils, visualization
from src.spatial import build_spatial_graph, validate_graph, spatial_lr
settings = utils.load_settings()
params = settings['params']
seed = params['seed']
sc.settings.set_figure_params(dpi=params['visualization']['figure_dpi'], facecolor='white')

In [ ]:
run = utils.begin_stage(settings, 'adata_qc.h5ad')
adata = preprocessing.load_all_samples(settings['raw_dir'], settings['samples'])
input_spots = adata.n_obs
adata = preprocessing.calculate_qc_metrics(adata)
qc = params['preprocessing']
adata = preprocessing.filter_spots(adata, min_genes=qc['min_genes'], min_cells=qc['min_cells'], max_pct_mt=qc['max_pct_mt'])
if set(adata.obs['sample']) != {sample['name'] for sample in settings['samples']}:
    raise ValueError('QC removed an entire sample; inspect filtering before proceeding')
adata = preprocessing.normalize_data(adata)
adata = preprocessing.select_hvg(adata, n_top_genes=qc['n_top_genes'], batch_key='sample')
assert adata.obs_names.is_unique
print(f'QC: {input_spots} -> {adata.n_obs} spots; {adata.n_vars} genes')

In [ ]:
summary = utils.sample_score_summary(adata, ['total_counts', 'n_genes_by_counts', 'pct_counts_mt'])
summary.to_csv(run / 'tables' / 'sample_qc.csv', index=False)
visualization.plot_qc_metrics(adata, save_path=run / 'figures' / 'qc.png')
plt.show()
utils.save_checkpoint(adata, run / 'processed' / 'adata_qc.h5ad')
summary